# 1단계 EDA 및 도시별 샘플링

Yelp 원본 JSON에서 분석 대상 도시의 레스토랑 리뷰를 추출하고, 감성 라벨을 만든 뒤 도시별 15,000건 샘플 CSV를 저장한다. 마지막 섹션에서는 샘플 품질을 확인하는 표와 그래프를 `output/`에 함께 저장한다.

노트북 위치: `notebooks/01_eda_sampling/`


In [ ]:
from google.colab import drive
from google.colab import files
from pathlib import Path

# 1. 내 구글 드라이브 연동 (결과물을 저장하기 위함)
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/ml_project/Team-6')
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
FIGURE_DIR = PROJECT_ROOT / 'output' / 'figures'
TABLE_DIR = PROJECT_ROOT / 'output' / 'tables'
for directory in [INTERIM_DIR, FIGURE_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# 2. 캐글 API 토큰(kaggle.json) 업로드
print("캐글에서 다운받은 kaggle.json 파일을 업로드한다.")
files.upload()

## 1. Colab 및 Kaggle 환경 준비

구글 드라이브를 마운트하고 Kaggle API 토큰을 등록한다. 원본 JSON은 코랩 임시 저장소에 압축 해제하여 읽기 속도를 확보한다.


In [ ]:
# 캐글 폴더 세팅 및 권한 부여
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Yelp 데이터셋 다운로드
!kaggle datasets download -d yelp-dataset/yelp-dataset

# 코랩 임시 공간에 압축 풀기 
!unzip -q yelp-dataset.zip -d /content/yelp_data
print("데이터 다운로드 및 압축 해제 완료!")

In [ ]:
import pandas as pd

# 비즈니스 데이터 불러오기
biz_path = '/content/yelp_data/yelp_academic_dataset_business.json'
biz_df = pd.read_json(biz_path, lines=True)

# 'Restaurants' 카테고리 필터링 (결측치 처리 포함)
biz_df = biz_df[biz_df['categories'].fillna('').str.contains('Restaurants')]

# 필라델피아 도시 필터링
target_city = 'Philadelphia'
biz_city = biz_df[biz_df['city'] == target_city]

# 추출할 식당들의 고유 ID 리스트 확보
target_biz_ids = set(biz_city['business_id'].tolist()) # 검색 속도를 위해 set으로 변환
print(f"{target_city} 내 대상 식당 개수: {len(target_biz_ids)}개")

## 2. Philadelphia 샘플 생성

비즈니스 JSON에서 Philadelphia 레스토랑 `business_id`를 먼저 추출한 뒤, 리뷰 JSON을 청크 단위로 읽어 해당 식당 리뷰만 남긴다.


In [ ]:
review_path = '/content/yelp_data/yelp_academic_dataset_review.json'
chunk_size = 100000
review_chunks = []

# 10만 줄씩 잘라서 읽기
for chunk in pd.read_json(review_path, lines=True, chunksize=chunk_size):
    # 대상 식당 ID에 해당하는 리뷰만 남기기
    filtered_chunk = chunk[chunk['business_id'].isin(target_biz_ids)]
    review_chunks.append(filtered_chunk)

# 모아둔 조각들을 하나의 데이터프레임으로 병합
review_subset = pd.concat(review_chunks, ignore_index=True)
print(f"필터링된 총 리뷰 개수: {len(review_subset)}개")

In [ ]:
# 1. 3점 리뷰(중립) 제거
review_subset = review_subset[review_subset['stars'] != 3]

# 2. 4~5점은 1(긍정), 1~2점은 0(부정)으로 매핑
review_subset['is_positive'] = review_subset['stars'].apply(lambda x: 1 if x > 3 else 0)

# 3. 15,000개 무작위 샘플링 (10,000개 이상 요건 충족)
final_dataset = review_subset.sample(n=15000, random_state=42)

print("최종 샘플링 데이터 크기:", final_dataset.shape)
display(final_dataset.head(3))

In [ ]:
# 내 구글 드라이브의 원하는 경로 지정
save_path = INTERIM_DIR / 'yelp_subset_philly_15k.csv'

# CSV 파일로 영구 저장
final_dataset.to_csv(save_path, index=False)

print(f"저장 완료. 경로: {save_path}")
print("코랩 세션이 초기화되어도 저장된 CSV에서 전처리를 이어갈 수 있다.")

In [ ]:
import pandas as pd

# 비즈니스 데이터 불러오기
biz_path = '/content/yelp_data/yelp_academic_dataset_business.json'
biz_df = pd.read_json(biz_path, lines=True)

# 'Restaurants' 카테고리 필터링 (결측치 처리 포함)
biz_df = biz_df[biz_df['categories'].fillna('').str.contains('Restaurants')]

# 투손(Tucson) 도시 필터링 적용
target_city = 'Tucson'
biz_city = biz_df[biz_df['city'] == target_city]

# 추출할 식당들의 고유 ID 리스트 확보
target_biz_ids = set(biz_city['business_id'].tolist())
print(f"{target_city} 내 대상 식당 개수: {len(target_biz_ids)}개")

In [ ]:
review_path = '/content/yelp_data/yelp_academic_dataset_review.json'
chunk_size = 100000
review_chunks = []

print(f"{target_city} 리뷰 데이터 청크 필터링 시작... (약 3~5분 소요 예상)")

# 10만 줄씩 잘라서 읽기
for chunk in pd.read_json(review_path, lines=True, chunksize=chunk_size):
    # 대상 식당 ID에 해당하는 리뷰만 남기기
    filtered_chunk = chunk[chunk['business_id'].isin(target_biz_ids)]
    review_chunks.append(filtered_chunk)

# 모아둔 조각들을 하나의 데이터프레임으로 병합
review_subset = pd.concat(review_chunks, ignore_index=True)
print(f"필터링된 총 리뷰 개수: {len(review_subset)}개")

## 3. Tucson 샘플 생성

동일한 기준으로 Tucson 레스토랑 리뷰를 필터링하고 3점 중립 리뷰를 제거한 뒤 15,000건을 샘플링한다.


In [ ]:
# 1. 3점 리뷰(중립) 제거
review_subset = review_subset[review_subset['stars'] != 3]

# 2. 4~5점은 1(긍정), 1~2점은 0(부정)으로 매핑
review_subset['is_positive'] = review_subset['stars'].apply(lambda x: 1 if x > 3 else 0)

# 3. 15,000개 무작위 샘플링
final_dataset = review_subset.sample(n=15000, random_state=42)

print("최종 샘플링 데이터 크기:", final_dataset.shape)

display(final_dataset.head(3))

In [ ]:
# 내 구글 드라이브의 원하는 경로 지정 (Tucson 파일명 적용)
save_path = INTERIM_DIR / 'yelp_subset_tucson_15k.csv'

# CSV 파일로 영구 저장
final_dataset.to_csv(save_path, index=False)

print(f"저장 완료. 경로: {save_path}")
print("투손 지역 15,000개 리뷰 데이터를 저장했다.")

In [ ]:
import pandas as pd

# 비즈니스 데이터 불러오기
biz_path = '/content/yelp_data/yelp_academic_dataset_business.json'
biz_df = pd.read_json(biz_path, lines=True)

# 'Restaurants' 카테고리 필터링 (결측치 처리 포함)
biz_df = biz_df[biz_df['categories'].fillna('').str.contains('Restaurants')]

# 뉴올리언스(New Orleans) 도시 필터링 적용
target_city = 'New Orleans'
biz_city = biz_df[biz_df['city'] == target_city]

# 추출할 식당들의 고유 ID 리스트 확보
target_biz_ids = set(biz_city['business_id'].tolist())
print(f"{target_city} 내 대상 식당 개수: {len(target_biz_ids)}개")

In [ ]:
review_path = '/content/yelp_data/yelp_academic_dataset_review.json'
chunk_size = 100000
review_chunks = []

print(f"{target_city} 리뷰 데이터 청크 필터링 시작... (약 3~5분 소요 예상)")

# 10만 줄씩 잘라서 읽기
for chunk in pd.read_json(review_path, lines=True, chunksize=chunk_size):
    # 대상 식당 ID에 해당하는 리뷰만 남기기
    filtered_chunk = chunk[chunk['business_id'].isin(target_biz_ids)]
    review_chunks.append(filtered_chunk)

# 모아둔 조각들을 하나의 데이터프레임으로 병합
review_subset = pd.concat(review_chunks, ignore_index=True)
print(f"필터링된 총 리뷰 개수: {len(review_subset)}개")

In [ ]:
# 1. 3점 리뷰(중립) 제거
review_subset = review_subset[review_subset['stars'] != 3]

# 2. 4~5점은 1(긍정), 1~2점은 0(부정)으로 매핑
review_subset['is_positive'] = review_subset['stars'].apply(lambda x: 1 if x > 3 else 0)

# 3. 15,000개 무작위 샘플링
final_dataset = review_subset.sample(n=15000, random_state=42)

print("최종 샘플링 데이터 크기:", final_dataset.shape)
display(final_dataset.head(3))

## 4. New Orleans 샘플 생성

New Orleans도 같은 파이프라인을 적용하여 도시별 비교가 가능한 입력 CSV를 만든다.


In [ ]:
# 내 구글 드라이브의 원하는 경로 지정 (New Orleans 파일명 적용)
save_path = INTERIM_DIR / 'yelp_subset_new_orleans_15k.csv'

# CSV 파일로 영구 저장
final_dataset.to_csv(save_path, index=False)

print(f"저장 완료. 경로: {save_path}")

In [ ]:
# 1. 코랩 리눅스 시스템에 나눔 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 2. 설치된 폰트를 Matplotlib의 기본 폰트로 설정
import matplotlib.pyplot as plt
plt.rc('font', family='NanumBarunGothic')

# 3. 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

## 5. 샘플 EDA 및 시각화 저장

각 도시 샘플의 타깃 분포, 텍스트 길이, 워드클라우드를 확인하고 PNG 파일을 `output/figures/`에 저장한다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

plt.rc('font', family='NanumBarunGothic')
plt.rcParams['axes.unicode_minus'] = False

BASE_STOPWORDS = {
    "food", "place", "restaurant", "good", "great", "time", "one",
    "really", "even", "came", "went", "got", "make", "will", "go",
    "ordered", "us", "back", "much", "well"
}


def save_figure(filename):
    path = FIGURE_DIR / filename
    plt.savefig(path, dpi=160, bbox_inches='tight')
    print(f"그래프 저장 완료: {path}")


def plot_city_eda(city_name, city_slug, input_file, extra_stopwords=None):
    file_path = INTERIM_DIR / input_file
    df = pd.read_csv(file_path)
    df['text_length'] = df['text'].astype(str).apply(len)

    summary_table = df.groupby('is_positive').agg(
        review_count=('review_id', 'count'),
        avg_stars=('stars', 'mean'),
        avg_text_length=('text_length', 'mean'),
    ).reset_index()
    summary_path = TABLE_DIR / f'eda_{city_slug}_summary.csv'
    summary_table.to_csv(summary_path, index=False)
    print(f"[{city_name}] 요약표 저장 완료: {summary_path}")
    display(summary_table)

    plt.figure(figsize=(8, 6))
    ax = sns.countplot(data=df, x='is_positive', hue='is_positive', palette=['#ff9999', '#66b3ff'], legend=False)
    plt.title(f'{city_name} 리뷰 만족도 분포', fontsize=16, fontweight='bold')
    plt.xlabel('만족도 (0: 불만족, 1: 만족)', fontsize=12)
    plt.ylabel('리뷰 개수', fontsize=12)
    for patch in ax.patches:
        height = patch.get_height()
        ax.text(patch.get_x() + patch.get_width() / 2, height + 50, f'{int(height):,}', ha='center', size=12)
    plt.tight_layout()
    save_figure(f'eda_{city_slug}_target_distribution.png')
    plt.show()

    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='is_positive', y='text_length', hue='is_positive', palette=['#ff9999', '#66b3ff'], legend=False)
    plt.title(f'{city_name} 만족도별 리뷰 텍스트 길이', fontsize=16, fontweight='bold')
    plt.xlabel('만족도 (0: 불만족, 1: 만족)', fontsize=12)
    plt.ylabel('리뷰 길이 (글자 수)', fontsize=12)
    plt.ylim(0, df['text_length'].quantile(0.95))
    plt.tight_layout()
    save_figure(f'eda_{city_slug}_text_length_boxplot.png')
    plt.show()

    pos_text = " ".join(review for review in df[df['is_positive'] == 1]['text'].astype(str))
    neg_text = " ".join(review for review in df[df['is_positive'] == 0]['text'].astype(str))
    custom_stopwords = set(STOPWORDS) | BASE_STOPWORDS | set(extra_stopwords or [])

    wc_pos = WordCloud(width=600, height=400, background_color='white', colormap='Greens', stopwords=custom_stopwords, max_words=100).generate(pos_text)
    wc_neg = WordCloud(width=600, height=400, background_color='white', colormap='Reds', stopwords=custom_stopwords, max_words=100).generate(neg_text)

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    axes[0].imshow(wc_pos, interpolation='bilinear')
    axes[0].set_title('만족(긍정) 핵심 키워드', fontsize=18, fontweight='bold')
    axes[0].axis('off')
    axes[1].imshow(wc_neg, interpolation='bilinear')
    axes[1].set_title('불만족(부정) 위기 키워드', fontsize=18, fontweight='bold')
    axes[1].axis('off')
    plt.tight_layout()
    save_figure(f'eda_{city_slug}_wordcloud.png')
    plt.show()


### 필라델피아 EDA

샘플의 타깃 불균형, 텍스트 길이, 긍정/부정 키워드를 확인한다.


In [ ]:
plot_city_eda('Philadelphia', 'philly', 'yelp_subset_philly_15k.csv')


### 투손 EDA

도시명이 워드클라우드에 과도하게 반영되지 않도록 Tucson 관련 단어를 stopword에 추가한다.


In [ ]:
plot_city_eda('Tucson', 'tucson', 'yelp_subset_tucson_15k.csv', extra_stopwords=['Tucson', 'tucson'])


### 뉴올리언스 EDA

New Orleans/NOLA 표현이 핵심 키워드를 가리지 않도록 도시명 관련 단어를 제외한다.


In [ ]:
plot_city_eda('New Orleans', 'new_orleans', 'yelp_subset_new_orleans_15k.csv', extra_stopwords=['New Orlean', 'New Orleans', 'NOLA', 'nola', 'new orleans', 'New', 'Orlean'])
